# p_1 — session bootstrap

**Run this first, every session.**

1. Mounts Google Drive and creates `MyDrive/p_1/{corpus,checkpoints,results}`
2. Clones (first run) or pulls (every run after) the GitHub repo into `/content/p_1`
3. Sets your git identity
4. Pushes a `heartbeat.txt` test commit to prove the Colab → GitHub pipeline

Needs the Colab Secret `GITHUB_TOKEN` (🔑 key icon in the left sidebar).


## 1 · Mount Google Drive + project folders

Drive is the project's permanent disk: anything saved under `MyDrive/p_1/`
survives Colab session deaths (they *will* happen — that's normal weather).


In [ ]:
# 1) Mount Google Drive + create the project folders
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/p_1'
for sub in ('corpus', 'checkpoints', 'results'):
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)

print('Drive mounted ✓')
print('Project folder ready:', BASE)
for sub in ('corpus', 'checkpoints', 'results'):
    p = os.path.join(BASE, sub)
    print(f'  {p}  ({len(os.listdir(p))} items)')


## 2 · GitHub: clone or pull the repo

First run in a session → clones `p_1` into `/content/p_1`.
Every later run → `git pull` to get the latest.

The token is read from Colab Secrets (`GITHUB_TOKEN`) and is **never
printed**: every git command runs with `check=True` but its output is
captured and masked before it hits the notebook — even on failure.


In [ ]:
# 2) GitHub: clone (first time) or pull (every session after)
import subprocess
from pathlib import Path
from google.colab import userdata

OWNER = 'YOURUSERNAME'        # <-- EDIT ONCE: your GitHub username
REPO  = '/content/p_1'

# --- token from Colab Secrets (the ONLY place it lives) ---
TOKEN = None
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
    print('GITHUB_TOKEN loaded from Colab Secrets ✓')
except Exception as e:
    print('✗ No Colab Secret named GITHUB_TOKEN found.')
    print('  Fix: click the 🔑 key icon in the left sidebar → Add a new secret')
    print('       name:  GITHUB_TOKEN')
    print('       value: your GitHub fine-grained token (Contents: Read & write)')
    print('  Then enable notebook access and re-run this cell.')
    print(f'  (error was: {type(e).__name__})')

def sh(args, cwd=None):
    """Run a command with check=True, but capture its output and print it
    MASKED — so git can never echo the token into the notebook output."""
    shown = ' '.join(map(str, args)).replace(TOKEN, '***')
    try:
        r = subprocess.run(args, cwd=cwd, text=True,
                           capture_output=True, check=True)
    except subprocess.CalledProcessError as e:
        out = ((e.stdout or '') + (e.stderr or '')).replace(TOKEN, '***')
        print('$', shown)
        print(out.rstrip())
        raise SystemExit(f'FAILED ({e.returncode}): {shown}') from None
    print('$', shown)
    out = ((r.stdout or '') + (r.stderr or '')).replace(TOKEN, '***')
    if out.strip():
        print(out.rstrip())
    return r

if TOKEN:
    if Path(REPO, '.git').exists():
        sh(['git', '-C', REPO, 'pull'])
        print('Repo updated ✓')
    else:
        sh(['git', 'clone', f'https://{TOKEN}@github.com/{OWNER}/p_1.git', REPO])
        print('Repo cloned ✓')
    print('Code lives at', REPO)
else:
    print('Skipped GitHub setup — add the GITHUB_TOKEN secret, then re-run.')


## 3 · Set git identity

Signs your commits. **Edit `GIT_NAME` / `GIT_EMAIL` once** — use the
email you want shown on github.com.


In [ ]:
# 3) Git identity — who signs your commits (EDIT ONCE)
GIT_NAME  = 'YOURNAME'         # <-- EDIT ONCE: your name
GIT_EMAIL = 'you@example.com'  # <-- EDIT ONCE: the email GitHub uses for you

if Path(REPO, '.git').exists():
    sh(['git', '-C', REPO, 'config', 'user.name',  GIT_NAME])
    sh(['git', '-C', REPO, 'config', 'user.email', GIT_EMAIL])
    print('Git identity set (edit GIT_NAME / GIT_EMAIL above if this is not you):')
    print('  user.name  =', GIT_NAME)
    print('  user.email =', GIT_EMAIL)
else:
    print('Repo not ready — run the GitHub cell above first.')


## 4 · Heartbeat: the test commit

Writes `results/heartbeat.txt`, commits it, pushes it to `main`.

**DONE WHEN: you see heartbeat.txt in the repo on github.com** — that
proves the whole Colab → Drive → GitHub pipeline works.


In [ ]:
# 4) Heartbeat: prove the Colab → GitHub pipeline with a test commit
import datetime

if Path(REPO, '.git').exists():
    stamp = datetime.datetime.now().isoformat(timespec='seconds')
    hb = Path(REPO, 'results', 'heartbeat.txt')
    hb.parent.mkdir(parents=True, exist_ok=True)
    hb.write_text(f'p_1 alive from Colab — {stamp}\n')
    print('Wrote', hb)

    sh(['git', '-C', REPO, 'add', 'results/heartbeat.txt'])
    sh(['git', '-C', REPO, 'commit', '-m', 'heartbeat from Colab'])
    sh(['git', '-C', REPO, 'push'])
    print()
    print('DONE WHEN: you see heartbeat.txt in the repo on github.com')
    print(f'  → https://github.com/{OWNER}/p_1/blob/main/results/heartbeat.txt')
else:
    print('Repo not ready — run the GitHub cell above first.')


## The 5-command git ritual — every session

```bash
git pull                      # 1. start of session: get latest
# ...work (edit files, run notebooks)...
git status                    # 2. see what changed
git add -A                    # 3. stage everything
git commit -m "what I did"    # 4. snapshot with a message
git push                      # 5. ship it (end of every session)
```

Rescue one-liners (undo last commit, diff, unstage, crash check):
see `GIT_CHEATSHEET.md` in the repo root.
